## Week 2 - Assignment 2 for COfDSAI Tutorial Markdown ##

For this week, need to first interpret the requests of the board members.
First do all the importing and initialisation.

In [19]:
# necessary imports
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from scipy.stats import norm
from load_industries import load_industry_returns, estimation_inputs

In [20]:
# want to run the load_industries.py file to get the needed data
R = load_industry_returns("30_Industry_Portfolios.csv")
mu, Sigma, names = estimation_inputs(R, start="2015-01", end="2024-12")

#print(mu, Sigma, names)


First we need to run the base case again so that we can compare to it.

In [ ]:

# initialise for the base case 
n = len(mu)
weq = 1/n
weq_v = weq*np.ones(30)
#weq_v

sigma2 = weq_v.T @ Sigma @ weq_v
sigma2
volatilityeq = 100 * np.sqrt(weq_v @ Sigma @ weq_v)

print("The upper bound of our variance is:", sigma2*100, "%")
print("With a volatility of: ", volatilityeq, "%")


wa = np.clip(w.value, 0, None)
wa = wa / wa.sum()

status = {}
def solve(problem, var, name):
    problem.solve()
    status[name] = problem.status
    if var.value is None:                        # infeasible / failed
        return None
    return np.clip(var.value, 0, None)

# --- base case: max return s.t. variance <= equal-weight variance -------
w = cp.Variable(n)
w_eq = np.ones(n) / n
base_cons = [cp.sum(w) == 1, w >= 0,
             cp.quad_form(w, Sigma) <= w_eq @ Sigma @ w_eq]
w_base = solve(cp.Problem(cp.Maximize(mu @ w), base_cons), w, "base")

# --- "good return" floor: at least 80% of the base-case return ----------
r_floor = 0.8 * (mu @ w_base)
USE_FLOOR = True



#  have n variables w
w = cp.Variable(n)
proba = cp.Problem(cp.Maximize(mu @ w),
                    [
                        cp.sum(w) == 1,
                        w >= 0,
                        cp.quad_form(w, Sigma) <= sigma2,
                    ])

proba.solve()

# get annual return and volatility (where volatility is the standard deviation)
returna = proba.value * 100
volatilitya = 100 * np.sqrt(wa @ Sigma @ wa)

# check if the solution is optimal
print("Status of solution: ", proba.status)
print("Expected Annual Return of Base Case Portfolio: ", returna, "%")
print("Annual Volatility of Base Case Portfolio: ", volatilitya, "%")

# look at the weights that were assigned to each stock in the portfolio
for i in range(n):
    if w.value[i] > 0.0001: # put a threshold, so if the weight of that stock is less than 0.0001 we ignore it
        print(names[i], round(100 * w.value[i], 2), "%") # print the weights of the invested stocks

The upper bound of our variance is: 3.0154289808169934 %
With a volatility of:  17.364990586858934 %
Status of solution:  optimal
Expected Annual Return of Base Case Portfolio:  19.232658942032668 %
Annual Volatility of Base Case Portfolio:  17.364990598603864 %
Smoke 14.4 %
Util 6.38 %
Servs 17.3 %
BusEq 60.35 %
Meals 1.57 %


Define the function that'll print all the metrics for these portfolios.

In [22]:
def metrics(wv):
    rp = R @ wv                                           # monthly portfolio returns
    wealth = np.concatenate([[1], np.cumprod(1 + rp)])    # W_0 = 1, W_1..W_T
    ret, vol = mu @ wv, np.sqrt(wv @ Sigma @ wv)
    return {
        "return %":   100 * ret,
        "vol %":      100 * vol,
        "ratio":      ret / vol,
        "max_dd %":   100 * np.max(1 - wealth / np.maximum.accumulate(wealth)),
        "worst12m %": 100 * min(np.prod(1 + rp[i:i + 12]) - 1 for i in range(T - 11)),
        "bad6 %":     100 * (-np.sort(rp)[:k]).mean(),    # avg of 6 worst months, as loss
        "5%VaR %":    100 * (ret - z * vol),              # normal-approx. 5% annual cutoff
        "max_w":      wv.max(),
        "n_eff":      1 / np.sum(wv ** 2),
    }

def print_portfolio_metrics(weights, portfolio_name, expected_return_val, volatility_val):
    """
    Calculates and prints the key metrics for a given portfolio.
    
    Args:
        weights (np.array): The optimized portfolio weights.
        portfolio_name (str): The name of the portfolio for display.
        expected_return_val (float): The calculated expected annual return.
        volatility_val (float): The calculated annual volatility.
    """
    # Filter out near-zero weights (e.g., < 0.1%) to avoid division by zero errors
    # and to ensure a clean calculation.
    threshold = 0.001
    clean_weights = np.array([w for w in weights if w > threshold])
    
    # Calculate the effective number of industries
    n_eff = 1 / np.sum(clean_weights ** 2)
    
    print(f"--- {portfolio_name} ---")
    print(f"Expected Annual Return:           {expected_return_val:.2f}%")
    print(f"Annual Volatility:                {volatility_val:.2f}%")
    print(f"Effective Number of Industries:   {n_eff:.2f}\n")
    
    print("Significant Stock Weights (>0.1%):")
    for i in range(len(weights)):
        if weights[i] > threshold:
            print(f"  {names[i]:<7} {weights[i]*100:.2f}%")
    print("\n" + "="*40 + "\n")


# Implementation and comparison

I decided to implement the Chair's wish (using the Sharpe ratio) and the Risk manager's wish. 
I define the "good" return so that it can be compared to.

In [23]:
# first define the "good" return
rgood = 0.8 * (mu @ wa)
print("The \"Good\" Annual Return: ", rgood*100, "%")

The "Good" Annual Return:  15.38612715362614 %


Now implement the Sharpe ratio based convex problem (both the original and reformulated to see which works). We consider there to be no risk-free rate (from assignment).
The first formulation should fail as it isn't convex, and the reformulation should work.


In [24]:
# first initialise the variables needed for metrics and formulation
rf = 0
T, n = R.shape
k = int(round(0.05 * T)) # 5% of the 120 months
z = norm.ppf(0.95) # the approximated normal at 5% = 1.645


wsncvx = cp.Variable(n)
probsncvx = cp.Problem(cp.Maximize((mu @ wsncvx - rf) / cp.sqrt(cp.quad_form(wsncvx, Sigma))),
                                [
                                    cp.sum(wsncvx) == 1,
                                    wsncvx >= 0,
                                    mu @ wsncvx >= rgood
                                ])

try:
    probsncvx.solve()
except cp.error.DCPError as e:
    print("Status of solution: Rejected by CVXPY")

probsncvx.is_dcp()

Status of solution: Rejected by CVXPY


False

Now we can do the reformulated version of the problem, and observe our returns.

In [31]:
# introduce the new decision variables for the reformulated Sharpe ratio
y = cp.Variable(n)
kappa = cp.Variable()

probsharpe = cp.Problem(cp.Minimize(cp.QuadForm(y, Sigma)), 
                                    [
                                        mu @ y - rf * kappa == 1,
                                        cp.sum(y) == kappa,
                                        y >= 0,
                                        kappa >= 0,
                                        mu @ y >= rgood * kappa
                                    ])
yval = probsharpe.solve()
wsharpe = None if yval is None else yval / kappa.value

ysharpe = np.clip(y.value / kappa.value, 0, None)
ysharpe = y / y.sum()

# get annual return and volatility (where volatility is the standard deviation)
returnsharpe = probsharpe.value * 100
volatilitysharpe = 100 * cp.sqrt(ysharpe @ Sigma @ ysharpe)


Now we can do the Risk manager's wish.

Finally, let's observe what the metrics for each problem are.

In [34]:
ports = {"base": wa, "sharpe": wsharpe}

print("\n--- Metrics per portfolio ---")
table = {nm: metrics(wv) for nm, wv in ports.items() if wv is not None}
cols = list(next(iter(table.values())).keys())
print(f"{'':14s}" + "".join(f"{c:>12s}" for c in cols))
for nm, m in table.items():
    print(f"{nm:14s}" + "".join(f"{m[c]:12.2f}" for c in cols))

print("\n--- Reformulation check: Sharpe ratio, convex vs direct ---")
print(f"convex reformulation: {table['sharpe']['ratio']:.4f}   "
      f"direct (SLSQP): {table['sharpe_direct']['ratio']:.4f}")

print("\n--- Risk manager's wish satisfied? (bad6 <= 8%) ---")
for nm, m in table.items():
    print(f"{nm:14s} {int(m['bad6 %'] <= 8 + 1e-6)}")

print("\n--- Top 5 weights per portfolio ---")
for nm, wv in ports.items():
    if wv is not None:
        top = np.argsort(-wv)[:5]
        print(f"{nm:14s}", ", ".join(f"#{i}:{wv[i]:.2f}" for i in top))


--- Metrics per portfolio ---


IndexError: tuple index out of range